In [28]:
# 1. 라이브러리 및 설정
!pip -q install optuna catboost

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [29]:
# 2. 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [30]:
# 3. 하이엔드 피처 엔지니어링 (Feature Explosion)
def ultimate_engineering(df):
    d = df.copy()

    # --- A. 기본 물리/생체 지표 ---
    d["height_cm"] = (d["Height(Feet)"] * 12 + d["Height(Remainder_Inches)"]) * 2.54
    d["weight_kg"] = d["Weight(lb)"] * 0.453592
    d["temp_c"] = (d["Body_Temperature(F)"] - 32) * (5/9)
    d["bmi"] = d["weight_kg"] / ((d["height_cm"]/100)**2)

    # --- B. 성별 기반 BMR (기초대사량) ---
    is_male = d['Gender'].isin(['M', 'Male'])
    d["bmr"] = np.where(is_male,
                        66.47 + (13.75 * d["weight_kg"]) + (5.0 * d["height_cm"]) - (6.76 * d["Age"]),
                        655.1 + (9.56 * d["weight_kg"]) + (1.85 * d["height_cm"]) - (4.68 * d["Age"]))

    # --- C. 파생 변수 (Interaction) ---
    # 칼로리 소모 공식(METs)에 기반한 논리적 상호작용
    d['intensity_proxy'] = d['BPM'] * d['Exercise_Duration']
    d['metabolic_equivalent'] = d['bmr'] * d['Exercise_Duration']
    d['thermal_stress'] = d['temp_c'] * d['Exercise_Duration']
    d['log_BPM'] = np.log1p(d['BPM'])

    # --- D. Feature Explosion (수치형 변수 간 모든 조합) ---
    # 핵심 변수 선정
    numeric_cols = ['Age', 'Exercise_Duration', 'BPM', 'temp_c', 'weight_kg', 'bmi', 'bmr']

    # 2차 상호작용 (Poly)
    poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
    poly_feats = poly.fit_transform(d[numeric_cols])
    poly_cols = [f'poly_{i}' for i in range(poly_feats.shape[1])]
    d = pd.concat([d, pd.DataFrame(poly_feats, columns=poly_cols, index=d.index)], axis=1)

    # 비율 변수 (Ratios)
    for col1 in ['BPM', 'Exercise_Duration']:
        for col2 in ['weight_kg', 'Age']:
            d[f'{col1}_div_{col2}'] = d[col1] / (d[col2] + 1e-5)

    # --- E. 인코딩 ---
    le = LabelEncoder()
    d['Gender'] = le.fit_transform(d['Gender'])
    d['Weight_Status'] = le.fit_transform(d['Weight_Status'])

    return d

print("Feature Engineering 진행 중...")
train_df = ultimate_engineering(train)
test_df = ultimate_engineering(test)

X = train_df.drop(['ID', 'Calories_Burned'], axis=1)
y = train_df['Calories_Burned']
test_X = test_df.drop(['ID'], axis=1)

# 4. Feature Selection (Lasso를 이용한 노이즈 제거)
# 피처가 너무 많으면 오히려 RMSE가 오르므로, 선형 관계가 약한 피처 제거
print("Feature Selection (Lasso) 진행 중...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lasso = Lasso(alpha=0.1) # alpha 조절로 피처 개수 제어
lasso.fit(X_scaled, y)

selected_mask = lasso.coef_ != 0
selected_features = X.columns[selected_mask]
print(f"선택된 피처 개수: {len(selected_features)} / {X.shape[1]}")

X = X[selected_features]
test_X = test_X[selected_features]

Feature Engineering 진행 중...
Feature Selection (Lasso) 진행 중...
선택된 피처 개수: 23 / 64


In [31]:
# 5. Optuna를 활용한 모델 튜닝 함수
def tune_model(model_name, X, y, n_trials=20):
    def objective(trial):
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        rmse_scores = []

        if model_name == 'cat':
            params = {
                'iterations': trial.suggest_int('iterations', 1000, 3000),
                'depth': trial.suggest_int('depth', 4, 10, 20),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 10),
                'random_seed': 42,
                'verbose': 0,
                'loss_function': 'RMSE'
            }
            model = CatBoostRegressor(**params)

        elif model_name == 'xgb':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
                'max_depth': trial.suggest_int('max_depth', 4, 10, 20),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'random_state': 42,
                'n_jobs': -1
            }
            model = XGBRegressor(**params)

        elif model_name == 'lgbm':
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'random_state': 42,
                'n_jobs': -1,
                'verbose': -1
            }
            model = LGBMRegressor(**params)

        # CV 검증
        for train_idx, val_idx in kf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse_scores.append(np.sqrt(mean_squared_error(y_val, preds)))

        return np.mean(rmse_scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    print(f"Best {model_name} RMSE: {study.best_value:.4f}")
    return study.best_params

In [32]:
# 6. Seed Averaging 학습 함수
def train_seed_avg(model_cls, params, X, y, test_X, seeds=[42, 2023, 9999]):
    oof_total = np.zeros(len(X))
    test_pred_total = np.zeros(len(test_X))

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for seed in seeds:
        params['random_state'] = seed  # 모델별 시드 파라미터 이름 확인 필요 (Cat: random_seed)
        if 'cat' in str(model_cls).lower():
             params['random_seed'] = seed
             del params['random_state']

        for train_idx, val_idx in kf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

            model = model_cls(**params)
            if 'cat' in str(model_cls).lower():
                model.fit(X_tr, y_tr, verbose=0)
            else:
                model.fit(X_tr, y_tr)

            oof_total[val_idx] += model.predict(X_val) / len(seeds)
            test_pred_total += model.predict(test_X) / (len(seeds) * 5) # 5 folds

    return oof_total, test_pred_total

In [33]:
# --- 실행 파이프라인 ---

# A. 모델별 하이퍼파라미터 튜닝 (시간 관계상 10회씩만. 실제로는 50회 이상 권장)
print("\n>>> Hyperparameter Tuning Start...")
best_cat_params = tune_model('cat', X, y, n_trials=50)
best_xgb_params = tune_model('xgb', X, y, n_trials=50)
best_lgbm_params = tune_model('lgbm', X, y, n_trials=50)

# B. Seed Averaging을 통한 예측 (Out-of-Fold 생성)
print("\n>>> Seed Averaging Prediction Start...")
cat_oof, cat_test = train_seed_avg(CatBoostRegressor, best_cat_params, X, y, test_X)
xgb_oof, xgb_test = train_seed_avg(XGBRegressor, best_xgb_params, X, y, test_X)
lgbm_oof, lgbm_test = train_seed_avg(LGBMRegressor, best_lgbm_params, X, y, test_X)

# C. Stacking (메타 모델 학습)
print("\n>>> Stacking (Meta-Model) Start...")
# 1단계 예측값을 새로운 피처로 사용
X_stack_train = pd.DataFrame({'Cat': cat_oof, 'XGB': xgb_oof, 'LGB': lgbm_oof})
X_stack_test = pd.DataFrame({'Cat': cat_test, 'XGB': xgb_test, 'LGB': lgbm_test})

# 메타 모델 (Ridge 사용 - 과적합 방지)
meta_model = Ridge(alpha=10)
meta_model.fit(X_stack_train, y)

# 최종 예측
final_predictions = meta_model.predict(X_stack_test)
final_rmse = np.sqrt(mean_squared_error(y, meta_model.predict(X_stack_train)))

print(f"\n🔥 Ultimate Stacking RMSE: {final_rmse:.5f}")


>>> Hyperparameter Tuning Start...
Best cat RMSE: 1.0142
Best xgb RMSE: 1.3756
Best lgbm RMSE: 1.6132

>>> Seed Averaging Prediction Start...

>>> Stacking (Meta-Model) Start...

🔥 Ultimate Stacking RMSE: 0.88669


In [34]:
# 7. 제출 파일 생성
submission = pd.read_csv('sample_submission.csv')
submission['Calories_Burned'] = final_predictions
submission.to_csv('./submit.csv', index = False)
print("submission_ultimate.csv saved!")

submission_ultimate.csv saved!
